# nl2cuda-kernel-agent — Colab 驱动入口

在 Colab（运行时选 **T4 GPU**）中依次运行下面的单元格。

本 notebook 只做三件事：拉取仓库、探测环境、跑基础设施自检（正确性验证器 + 计时基准器）。

**协作方式**：运行后把输出贴回给助手，助手改代码 push，你再 pull 重跑。

## 1. 拉取 / 更新仓库
首次运行会 clone；之后再运行会 pull 最新代码。

In [ ]:
import os

REPO_URL = 'https://github.com/SilenceWanna/nl2cuda-kernel-agent.git'
REPO_DIR = '/content/nl2cuda-kernel-agent'

if os.path.isdir(REPO_DIR):
    print('仓库已存在，执行 git pull ...')
    !cd {REPO_DIR} && git pull --ff-only
else:
    print('clone 仓库 ...')
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print('cwd =', os.getcwd())
!git log --oneline -3

## 2. 环境探测
确认 GPU 型号、CUDA / PyTorch 版本、TF32 支持等（对应 `scripts/probe_env.py`）。

In [ ]:
!python scripts/probe_env.py

## 2.5 编译链路冒烟测试（阶段1）
编译最小 `smoke.cu`（向量加法）并验证正确，确认 nvcc + cpp_extension + sm_75 编译链路可用。
这一步通过后，再编译真正的 RBF kernel 就能把"编译环境问题"和"kernel 逻辑问题"分开排查。

## 3. 正确性验证器自检
用参考实现验证参考实现自身，应当全 **PASS**（误差 ~0）。
这验证 `tests/verify.py` 在真实 GPU + 当前形状（默认 2048²，由 `RBF_SIZE` 控制）下可运行。

In [ ]:
!python tests/verify.py

## 4. 计时基准器自检
候选 = 未编译的参考实现，baseline = `torch.compile(参考)`。
预期 baseline 更快，故 candidate 加速比 <1、判 **FAIL** —— 这是正常的，
本步只验证计时管线（CUDA events / 几何均值 / CV / 前反向分离）可运行。

> 首次运行含 `torch.compile` 编译，耗时较久，请耐心等待。

In [ ]:
!python benchmarks/bench.py